# Kaggriculture: notes from replay hunting

I started this notebook as a collection of engine notes. It gradually
turned into a record of how I have been changing my own agent: download
the current leaders, work out which moves repeat, turn the repeatable
part into a local agent, and try to beat it from both seats.

That distinction matters. A replay is one match, not source code. The
losing player may be excellent, and the winner may only have had the
favorable seed or market order. I use both sides as evidence and do not
label every losing action as a mistake.

Kaggriculture is a 720-turn, two-player simulation. The leaderboard is a
skill rating based on wins, losses and ties; a larger coin margin does
not directly buy more rating. Local bank totals and ladder rating answer
different questions.

The first half of the notebook records the mechanics and earlier public
strategy families. Section 4 is the working diary: c14, Hamburger's
market-timing idea, c15, the next leaderboard refresh, and the c16
candidate that came out of it.

### Public work I used

| Author / notebook | What I took from it |
| --- | --- |
| [Bovard — Getting Started](https://www.kaggle.com/code/bovard/kaggriculture-getting-started) | Agent contract |
| [Georgy Mamarin — Visualized](https://www.kaggle.com/code/georgymamarin/kaggriculture-visualized-what-every-crop-pays) | Mechanics and market charts |
| [Roman Rozen — Barnyard Economist](https://www.kaggle.com/code/romanrozen/strong-statr-baseline-agent-lb-950) | Melon timing and job-value framing |
| [Roman Tamrazov — Hamburger](https://www.kaggle.com/code/romantamrazov/kaggriculture-hamburger) | Staged mixed herds and clone-aware market timing |
| [Pilkwang Kim — Scenario-Aware](https://www.kaggle.com/code/pilkwang/kaggriculture-scenario-aware-economic-policy) | Economic scheduler lineage |
| [Kun Zhang — C03/C04/C05](https://www.kaggle.com/code/beicicc/kaggriculture-c05-mid-herd-10) | Herd target experiments |
| [prvsiyan — Frontier Lab](https://www.kaggle.com/code/prvsiyan/kaggriculture-frontier-lab-high-score-visuals) | Cross-play gates |
| Public leaderboard replays | Repeated schedules, opponent responses and market timing |

The notebook embeds the exact candidate used in the final local gate.
Running it creates `main.py` and `submission.tar.gz`; it does not upload
anything by itself.


## 0. Setup

Uses the competition environment from `kaggle-environments` (see the competition data kit `AGENTS.md` / `README.md`).


In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

try:
    installed_ke = version("kaggle-environments")
except PackageNotFoundError:
    installed_ke = "0"

# Kaggle's base notebook image may lag the competition runner. The replay-tape
# policies in this notebook require the current Kaggriculture mechanics.
if tuple(map(int, installed_ke.split(".")[:3])) < (1, 32, 2):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle-environments>=1.32.2"
    ])

import kaggle_environments
print("kaggle-environments", kaggle_environments.__version__)

from kaggle_environments import make

BOARD, DAYS, TURNS_PER_DAY = 10, 30, 24
EPISODE_STEPS = DAYS * TURNS_PER_DAY  # 720
STARTING_MONEY = 3000


## 1. The game in one screen

| Knob | Default | Why it matters |
| --- | ---: | --- |
| Turns | 720 (30 x 24) | Long-horizon capital plan |
| Start cash | $3000 | Opening buys compete with each other |
| Land | NE $1k, SW $2k, SE $4k | Top public leaders almost always take **NE+SW**, rarely SE |
| Hire cost | fib(n) per extra hand **today** | First ~10 hands are cheap; 12+ gets expensive fast |
| Shed | 100 non-seed items | Overflow is destroyed — you must sell / liquidate |
| Win | Most **bank** coins | Unsold inventory does **not** count |
| Ladder | Skill rating | **Only** W/L/T; coin margin is irrelevant for Elo |

### Action economy beats the crop table

Each unit (farmer + hired hands) gets **one field op per turn**.  
A high-margin crop that costs two extra walks often loses to a weaker crop next to the shed.

**Rule of thumb:** hiring about 10 hands costs roughly **$143 for a full day** of parallelism. Under-hiring early is usually more expensive than over-hiring inside the cheap Fibonacci region.


In [ ]:
def fib(n):
    a, b = 1, 1
    for _ in range(n):
        a, b = b, a + b
    return a


def hire_day_cost(n_hands):
    # total cost to hire n hands starting from 0 hires today (mult = 1)
    return sum(fib(i) for i in range(n_hands))


ns = np.arange(1, 17)
costs = [hire_day_cost(int(n)) for n in ns]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(ns, costs, "o-", lw=2, color="#0f766e")
ax.axvline(10, color="#f59e0b", ls="--", label="~10 hands (end of cheap region)")
ax.axvline(12, color="#dc2626", ls="--", label="~12 hands (steep)")
ax.set(
    xlabel="Hands hired today",
    ylabel="Total hire cost (coins)",
    title="Fibonacci hire curve (default mult=1)",
)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

pd.DataFrame({"hands": ns, "day_cost": costs}).query("hands in [6, 8, 10, 11, 12, 14]")


## 2. Engine details that separate weak agents from strong ones

### 2.1 Melon yield window (trust the engine, not only the overview table)

For one-time crops, watering during a bonus window adds yield immediately:

- Melon `max_yield_day = 12` implies window start `ceil(12/2) = 6` → ages **6..12**
- A melon starts at `yield_units = 1`
- Each watered day in-window: **+1** (or **+2** if fertilized)
- Cap is **6**, so full water reaches max around **age 10**, not 12

**16 tiles x 6 = 96 melons** when execution is perfect.  
Missing water on days 6-10 is a common silent leak (for example ~70 units instead of ~96).

### 2.2 `SELL` only sees the shed

- `HARVEST` puts items into the **unit inventory**
- `SELL` spends items from the **shed**

If you harvest 90 melons and never `DROP` while shed-adjacent, the market may not see them until end-of-day auto-drop — often **too late** to fund same-turn land / animal buys.

### 2.3 Fertilizer can be sold

Competition text emphasizes buying fertilizer; the engine's generic `SELL` path still accepts it.  
Each animal generates fertilizer daily → a real sidecar income stream until the market is flooded.

### 2.4 Ladder scoring

Rating updates from **win / loss / tie only**.  
A 140k bank that loses still loses rating. Local mean bank vs `starter` is a useful filter, not the objective.


In [ ]:
# Illustrative melon glut curve (public MARKET_PARAMS style, glut side only)
I0, BASE, T = 10000, 250, 300
above_target = 3.6  # melon above_target


def melon_price(inv):
    x = max(0.0, inv - I0)
    amp = above_target * BASE / (T ** 2)
    return max(1, int(round(BASE - amp * x * x)))


units_per_tile = 6
tiles = np.arange(1, 31)
marginal = []
inv = I0
for t in tiles:
    rev = 0
    for _ in range(units_per_tile):
        p = melon_price(inv)
        rev += p
        if p > 1:
            inv += 1
    marginal.append(rev - 80)  # minus seed cost

cum = np.cumsum(marginal)
best = int(tiles[np.argmax(cum)])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].bar(tiles, marginal, color=["#0f766e" if v > 0 else "#cbd5e1" for v in marginal])
axes[0].axhline(0, color="#334155", lw=1)
axes[0].set(
    title="Marginal net value of the next melon tile",
    xlabel="Tile # sold into one glut path",
    ylabel="Revenue - seed",
)
axes[1].plot(tiles, cum, lw=2.5, color="#0f766e")
axes[1].scatter([best], [cum[best - 1]], s=80, color="#dc2626", zorder=3, label=f"peak ~{best} tiles")
axes[1].set(title="Cumulative net (labor ignored)", xlabel="Melon tiles", ylabel="Cumulative net coins")
axes[1].legend()
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print({
    "illustrative_peak_tiles": best,
    "note": "Labor and rival sales move the real optimum; public leaders often use ~6-16 melons, not a full 25-tile dump.",
})


## 3. Strategy clusters in the public meta

| Cluster | Opening | Peak farm | Strengths | Failure mode |
| --- | --- | --- | --- | --- |
| **A. Pure cow ranch** | 3-10 cows, little crop | ~10 cows, often NE only | Punishes soft / crop-only bots | Milk wars collapse banks; coin-flip H2H |
| **B. Melon IPO** | 16 melons + 2 cows | Day-10 dump to 12c+8s and 3 quads | Huge capital spike when uncontested | Second melon dumper hits the $1 floor |
| **C. Staged economic herd (C0x)** | **3 cows + 1 sheep** floor | Stages **4 to 8/10 to 14/15** | Strongest *public code* family | Everyone forks it → correlated losses |
| **D. Adaptive leader style** | 3c+1s, ~6 melons | Dynamic cow/sheep/goose mix | Rebalances when milk is contested | Harder to clone cleanly |
| **E. Stable efficiency tape** | Repeated elite trajectory | Tight 8c/6s production and labor schedule | Reproducible across opponents; current best live result | Can become correlated if widely copied |

### Approximate local strength ordering (sandbox H2H)

```text
starter << pure cow << melon IPO << C03 << C05 << Radiant routers << c11/c12/c13 << c14
```

**Caveat:** a bot that prints 100k-170k vs `starter` can still sit mid-ladder, because ladder opponents are other strong farms.


## 4. Replay diary: from c14 to the current common meta

### 4.1 The first useful replay rule

My first mistake was treating the winner's tape as the target. That is
too noisy. I now look for three things before copying anything:

1. Does the same field schedule appear against several opponents?
2. Is the market schedule also stable, or is it reacting to the match?
3. Does the distilled tape beat the previous agent from both seats?

The earlier senkin13 refresh passed that test unusually cleanly: the
exact 720-turn field and market schedule appeared in five games. It
ended near 8 cows, 6 sheep and 7 strawberry plots, and became c14 after
I added an observation-driven final-eight-turn cleanup routine.

c14 beat c11, c12 and c13 27-3 each in the paired-seat gate and reached
2182.2 on its standalone ladder run. That was the point where replay
stability became more useful to me than selecting a spectacular single
score.


### 4.2 Hamburger changed the question from *what to build* to *when to sell*

The newer Hamburger notebook did not discover a radically different
mature farm. Its best branch, `Clone Quad H1`, checks whether the two
public farms remain nearly identical. After two close checkpoints it
looks one turn ahead in its own schedule and sells one available
premium line—melon, strawberry, milk or wool—before the expected
shared-market dump.

That is a small wrapper with a large mirror-match effect. Hamburger
reported 6-0 against its anchor, with a mean margin of +1,865.7. I
moved the same one-turn front-run onto the stronger refreshed Senkin
schedule. That hybrid became c15.


In [ ]:
c15_gate = pd.DataFrame([
    ["raw refreshed Senkin", 14, 2, 0, 2249.8],
    ["Vlad stable", 9, 7, 0, 725.5],
    ["Hamburger Clone Quad H1", 16, 0, 0, 2083.9],
    ["c14", 16, 0, 0, 2300.2],
    ["Batuhan rank-one replay", 12, 0, 0, 2538.3],
    ["Chloe common tape", 12, 0, 0, 2431.1],
    ["Kenmatsu anchor", 12, 0, 0, 2735.2],
], columns=["opponent", "wins", "losses", "ties", "mean_margin"])
c15_gate


The important row is the first one. c15 went 14-2 against the same
refreshed Senkin tape without the wrapper. That convinced me the
timing rule was doing real work rather than merely decorating a
stronger base.

### 4.3 Refreshing the leaders—and catching my own downloader mistake

On 3 August I pulled five replays for each displayed top-five team:

```bash
python scripts/download_top_replays.py --top 5 --per-team 5 \
    --leader-submission-only --force
```

The `--leader-submission-only` part was added after a bad first pass.
Simulation teams can have two active submissions. My original script
merged both episode lists and picked the newest games, which sampled
Tran H Hoang's and Knight of Favonius's newer *lower-rated* agents,
not the submissions responsible for their displayed leaderboard
scores. The corrected run matches `publicScore` to the displayed team
score before choosing episodes.

The leaderboard moved while I was doing this, so the numbers below
are a timestamped snapshot, not permanent ranks.


In [ ]:
replay_snapshot = pd.DataFrame([
    [1, "Tran H Hoang", 2858.3, 55203154, 5, 5, "5-0-0", "8c/5s, 5 strawberry, 12 hands"],
    [2, "Knight of Favonius", 2778.8, 55208294, 4, 4, "3-2-0", "8c/5s, 5 strawberry, 12 hands"],
    [3, "VN-Orion", 2735.3, 55207137, 1, 1, "4-1-0", "8c/5s, 5 strawberry, 12 hands"],
    [4, "Superallen001", 2717.9, 55207089, 1, 1, "2-3-0", "8c/5s, 4-5 strawberry, 12 hands"],
    [5, "ömer kiraz", 2700.9, 55210461, 3, 3, "4-0-1", "8c/5s, 5 strawberry, 12 hands"],
], columns=[
    "rank", "team", "rating", "submission_id", "field_variants",
    "market_variants", "sample_record", "mature_farm",
])
replay_snapshot


The striking result was not one clever move. It was convergence.
All five teams were using versions of the same 8-cow/5-sheep,
12-hand, three-quadrant plan. Many tapes from different teams differed
on only 2-5 field turns and 2-6 market turns. By contrast, this family
differed from c15 on about 226 field turns and 402 market turns.

Tran and Knight had several variants, so I did not use their highest
scoring replay as the new base. VN-Orion repeated one exact full tape
in all five games. Superallen also repeated one tape, and the two
stable versions are only four field turns and three market turns
apart. VN-Orion was the cleaner source for a standalone test.

I kept c15's clone-aware premium front-run and terminal cleanup, but
replaced its base schedule with the stable VN-Orion tape. That is the
c16 candidate below.


In [ ]:
c16_gate = pd.DataFrame([
    ["c15", 14, 2, 0],
    ["Hamburger Clone Quad H1", 15, 1, 0],
    ["c14", 11, 1, 0],
    ["Vlad stable", 11, 1, 0],
    ["refreshed Senkin", 11, 1, 0],
    ["Batuhan rank-one replay", 11, 1, 0],
    ["Superallen common-meta tape", 7, 7, 2],
], columns=["opponent", "wins", "losses", "ties"])
c16_gate


The 7-7-2 control is reassuring rather than disappointing: the two
independently sampled common-meta tapes behave like the same policy.
Against the previous families, the new schedule has a clear edge.

### 4.4 Why identical uploads can show very different ratings

I also chased what looked like a packaging bug. The standalone c14
scored 2182.2, while a notebook-produced c14 initially showed 1210.1.
I downloaded the notebook's actual server artifact and compared it
byte for byte. Both `main.py` files had SHA-256
`569cf2d20e3b37c3805c2a4ca7c4e5728eaa85df85c78392cdc3df77c5ddc17b`.

They really were the same agent. Kaggle had simply created a new
rating instance. The standalone copy accumulated 89 public games;
notebook v9 had 26 before it was displaced. While I was checking,
v9 moved from 1210.1 to 1865.6 without a code change. Early opponents,
seeds and seats can put identical copies on very different rating
paths, and only the latest two team submissions continue receiving
games.


In [ ]:
openings = pd.DataFrame([
    {"name": "Pure cow open", "melon_seeds": 0, "cows": 3, "sheep": 0, "notes": "expand cows later"},
    {"name": "Melon IPO open", "melon_seeds": 16, "cows": 2, "sheep": 0, "notes": "dump around day 10"},
    {"name": "C0x / radiant open", "melon_seeds": 6, "cows": 3, "sheep": 1, "notes": "mixed option value"},
    {"name": "Nishchal-like open", "melon_seeds": 9, "cows": 4, "sheep": 0, "notes": "early dairy density"},
])
openings["seed_cost"] = openings.melon_seeds * 80
openings["animal_cost"] = openings.cows * 400 + openings.sheep * 500
openings["rough_spend"] = openings.seed_cost + openings.animal_cost
openings["cash_left_from_3000"] = STARTING_MONEY - openings.rough_spend
openings


## 5. Live mini-simulations

### 5.1 Environment smoke test


In [ ]:
def run_pair(a="starter", b="starter", seed=0, steps=720):
    env = make("kaggriculture", configuration={"episodeSteps": steps, "seed": seed}, debug=False)
    env.run([a, b])
    final = env.steps[-1]
    return {
        "seed": seed,
        "r0": final[0].reward,
        "r1": final[1].reward,
        "s0": str(final[0].status),
        "s1": str(final[1].status),
    }

pd.DataFrame([run_pair("starter", "random", seed=s) for s in range(3)])


### 5.2 Built-in baseline matrix

Expect `starter` to beat `random` and `pass` on average. Your bot should crush `starter` locally before you overfit the public leaderboard number.


In [ ]:
rows = []
for a, b in [("starter", "random"), ("starter", "pass"), ("random", "pass")]:
    for seed in range(2):
        rows.append({"a": a, "b": b, **run_pair(a, b, seed=seed)})
pd.DataFrame(rows)


## 6. The build ladder I actually followed

I began with agents that merely survived validation, then moved through
pure cows, a melon capital event, staged mixed herds and public tape
routers. The last three steps mattered most:

- **c14:** choose a full schedule only after it repeats across opponents;
- **c15:** keep the schedule but front-run a clone's premium sale by one turn;
- **c16:** refresh the base when several leaders converge on a demonstrably
  stronger operating plan.

Every promotion used both seats. I keep the previous agent as an opponent
and add at least one unrelated public family. A high bank against
`starter` is a smoke test, not a promotion test.


## 7. Common bugs (high frequency)

| Bug | Symptom | Fix |
| --- | --- | --- |
| CARE ranked above melon WATER | Day 9 waters only part of the field; yield ~70 not ~96 | Water priority first in ages 6-12 |
| No DROP after HARVEST | Produce stuck in unit inventory; IPO underfunded | DROP when carrying melon/milk stacks |
| Dig melon tiles for pastures too early | Destroy fruit before harvest | Protect valuable melons until sold |
| Buy many cows day 0 with no feed reserve | Animals escape by day 2 | Reserve wheat cash before animal spam |
| Fixed 10 cows forever | Mirror banks collapse toward ~40k | Add sheep/strawberry/opponent routing |
| Optimize only mean bank vs starter | High local bank, mediocre Elo | H2H vs strong bots, both seats |
| Two near-identical active submits | Meta shift kills both | Diversify the second slot |


## 8. Local evaluation protocol

```text
1. Compile/import the exact packaged main.py
2. Self-play validation: both agents must finish DONE
3. Play the previous best on several seeds, both seats
4. Play at least one different public family
5. Include a near-mirror control when the meta has converged
6. Keep the result files, including losses
7. Only then build an archive
```

Shared-market games are not symmetric. A one-seat test can reverse the
apparent winner.


In [ ]:
def h2h(agent_a, agent_b, seeds=range(3), both_seats=True):
    """Quick head-to-head helper. agent_* may be callables or built-in names."""
    seats = (0, 1) if both_seats else (0,)
    rows = []
    for seed in seeds:
        for seat in seats:
            env = make("kaggriculture", configuration={"episodeSteps": 720, "seed": int(seed)})
            order = [agent_a, agent_b] if seat == 0 else [agent_b, agent_a]
            env.run(order)
            final = env.steps[-1]
            r0, r1 = final[0].reward, final[1].reward
            me, opp = (r0, r1) if seat == 0 else (r1, r0)
            rows.append({
                "seed": seed,
                "seat": seat,
                "me": me,
                "opp": opp,
                "win": me > opp,
                "status0": str(final[0].status),
                "status1": str(final[1].status),
            })
    df = pd.DataFrame(rows)
    summary = {
        "games": len(df),
        "wins": int(df.win.sum()),
        "win_rate": float(df.win.mean()),
        "mean_bank": float(df.me.mean()),
        "mean_opp": float(df.opp.mean()),
    }
    return df, summary


# Example using built-ins only (swap in your agent callables locally)
df, summary = h2h("starter", "random", seeds=range(2))
print(summary)
df


## 9. The strategy history in one line

```text
melon tutorials -> cow ranches -> staged mixed herds -> economic schedulers
  -> larger target herds -> tape routers -> stable c14 efficiency schedule
    -> c15 clone-aware sale timing -> c16 common 8c/5s/12-hand schedule
```

The endpoint alone never explained the full gain. c14 improved labor and
inventory timing. c15 exploited market order in mirror games. c16 changed
hundreds of turns while actually using one fewer sheep: the current
leaders are buying more labor and coordinating the whole route, not just
maximizing animal count.


## 10. Checklist before building the next archive

- [ ] Validation self-play ends `DONE` / `DONE`
- [ ] Harvested goods are dropped before a planned sale
- [ ] Final shed inventory is liquidated
- [ ] The replay source repeats across several opponents
- [ ] The selected replay belongs to the leaderboard-scoring submission,
  not merely the team's newest active submission
- [ ] Both seats were tested against the previous best
- [ ] A near-mirror control was included
- [ ] Losses were inspected instead of removed as “bad demonstrations”
- [ ] The exact packaged `main.py` was imported in the smoke test


## 11. Where I would look next

The top five have become highly correlated. That makes another wholesale
copy less interesting than finding the few turns where their variants
diverge. Tran and Knight change about twenty field turns in some matchups,
while VN-Orion and Superallen stay fixed. My next pass would line those
branches up against opponent farm signatures and test whether any are
deliberate counters rather than failed replay actions.

For now, c16 is the honest stopping point: a stable source, a clear local
edge over c15, and an equivalent result against an independently sampled
version of the same new meta.


## 12. Exact c16 artifact used for the local gate

This cell embeds the exact `c16_common_meta` source, writes it to
`/kaggle/working/main.py`, packages `submission.tar.gz`, imports that
generated file and runs a smoke game.

| Field | Value |
| --- | --- |
| Base replay | VN-Orion episode `89716627`, player 1 |
| Mature plan | about 8 cows, 5 sheep, 5 strawberry plots, 12 hands |
| Added controller | c15 clone-aware premium timing + terminal cleanup |
| Local result vs c15 | 14-2, paired seats |
| Ladder status | not submitted from this notebook |

Building the archive is deliberately separate from uploading it.


In [ ]:
import base64
import hashlib
import importlib.util
import tarfile
import zlib
from pathlib import Path

WORK = Path("/kaggle/working")
if not WORK.exists():
    WORK = Path.cwd()

MAIN_PATH = WORK / "main.py"
ARCHIVE_PATH = WORK / "submission.tar.gz"

# Exact c16 source from the local gate, compressed so the notebook stays readable.
_AGENT_B64_PARTS = [
    "eNqtfFm7okiQ6Hv/imO5DGjZg7u2KyIqiqCIovZXyiqibLKJ62+/qGetqu6ZOzPnRTIzIjIy9kj0fPv2TUjl/3qxHY5XpRfB0DRD"
    "f3EMM6lJDvdiCxtJdIOFg+JsXgTV0KUkd+As6cW0JE1xtRdH0RRd/vPbt29/KJppWM4Lz9lSPvs2Egzz+Pa8tQ397fmkKvwff6xo"
    "CkbQl+pj6U/V4EQbuK/8KUoBK8Eetg086f3JF3P3SVEC/ngJ/v5DSFq7CllbpAzuPNC11c5CViPV75O2uE/gzmVSrnWt7JVuG9ko"
    "Os1danLyIO/Qud30YLtW5goRvtA+kZmy1lUz+i7RmXTA6nlVTwwvxAZgLgTJwjOupQrxhj+ew9gNQoOhYq2bRnO3bqIZYUcsW/zq"
    "WGhjl5LjUlqEGjNJh+laQi/sc328YPYTW90kxWm+ymHAKWGsM+dtrq8RHemUZXFncsiEQsnKqq4y8dW6mrQ4m5FQ6dYSZuH6ZVHS"
    "wlTPhIu7xPkMdSVnntzFWzZyWoY4orxhCnvwEK1tu0sH5bJJy056LGWpC7m91dCydQb5WPioO8VtZYCVDY3cOg1W9ZcVstErpsP5"
    "6DhaNdrhY23PUcJw3Jm7CJYSpF0stXGrlVvvglzzTiPkhxYu36Er6el+fb5RS7dCDFCHKGbddI/d3+CeHwEH+cYEqx92MMgMxyUT"
    "z4Y2cj/EgURJGyD6sqgBToUXmHDDh9tIQ2dy5iB8SxZrvYhHH6OKT+49mpkJRR085+NLkSk3mHOmGdHCOtQZc25B6OfKe/BEDXte"
    "bF7bbK+HHCTVZLYRleC2X/MArVc6JcdOKVYw1RiUbm7bWVbrtBO3yG22hovMwManlr8AyIIlyRQwmdWkaZ+kRtVGlOiXSpkbNNzE"
    "fNjoVzI1INNQ7bgVbWiDfQNHa0JIkJaxKWKuBttjuo/WM7nOCoP63Tg7m4dyUDrcHVrpEz9lI+0zy1y7DX8XTsyBpGNLWJ7SKCaQ"
    "w3lUwvn4VMLgGD+VsRW67yDzUIUe1qfNudcwTGuZnhonM2LopWMym6kn9phhylRyy9OjDlkjc4dYl4wK+oWUGxSEGpCqDQ4Jq7R3"
    "a6fznDWZtUCd9SNqEIhxFWndmGY37LHQTWYixR67S/geIevsLAbhVnFlV8a0PV/mqxLVF+IypUqJmldY1MX92eMkTbzQyx6UDQvF"
    "Rb+623uAGW+HcReuzxeSJHnlRdnDYR+lwwcSh1vo+Rg9hyF85YSWrfoyOU0CCsoeCg29m8ZSw0SgMQpbHuanpbo2rsQAq4/YXXqW"
    "8MiINa71ah60j4FiN52IKz20UHXRTn2dgxgyWphGy8bERXaHOX2blhHFbxRL6bPdKqpLIm+4EWXSLtjn5ZzKDtpjMCLP8/vmMnz1"
    "EJ27Qp6njxrp1LTTWnUWpXF5Bi0noMRAc3EJqGV2sGqAzOlWGZF5HQxx8/4YkVXr6mr4yr8kQkq3UhhlUtPb3t83TrIrNm3lcnOY"
    "1KQTirRIe9Vgchmbb6hLT8/xG3YlLKY1NT6/pXIjeoZ4zX4cn1aINQTmJBSHFtSACautWmczUhMGs8GsiJ05RmrbwZjbMON2P5mv"
    "bLrMfBlbJi+pQlee8n2DLaKOXcMGx1ZJHgz7S4glqueQOG12m4PRVdgW0v3dfDcojpP4daLq+IJdKmFhlVgarStQi91a/ZmzOs1H"
    "fM3L3FacOsmEGgWhkRyX0y1bBLrDSpPwZ/3pbBbLWOntHIdcla55x0FHoQfAVm0OitfhKATTSfPsl6QbAyxkM5wttJbZymWU7lys"
    "xWCWYcsimT1DguaO29sWn2kOShPFWK22m9k+2UrBGd8c1+NtDz4dO6tZl7psdVXr+nUM0Ga8hzX3k8PJ2Bbxg7anDvN1WK66+bNR"
    "XCD2jYCgie2sYtN2r29DsVEVpIEQ3FRPfRyDqsXYblbABCcaGoSRKSWnl81DSx9OTGqnjQcmwkKJdV1KUNVarFpCI3ak406nqYGY"
    "itJsoibnWq15zTyuEpfiHgTC1gpiS2W5VyniI9ZqzPN4Y8JHwBANtufMnpZCW2qdEa32bLForeeDWqPS6O1m5jHBU0RYnFR2kZGW"
    "sJWkmc102pHUMDWeSLIOhY1lvGPj2JnUTC43TU/5sW3PBpltWm6faeyYm83gohPpG218FG3X8+V54Gtn4bjT+9ygeHBSk4gq8Axx"
    "HErd7FIqzazmWsliel3yxiksyQ99c1FhavahXrGWR6+8TVabcKOGHdsXvaMqGdGIlef9CbZPSpIyU+mx1TmF9XJjM+0jhF6kuvhh"
    "M6V7szm/3Zi3rrbLHhtWKLRUsGFuKS1idXwuJYZKrd6QoQxKWpcM1N/k6oC6sbpeK6oOsmgQliP4ou8neX5ANlqiNttmWitFCRXj"
    "FJ0NH0CqdpgURmx2g6vOqbLtFibAKV2axtlEfJO9XJB5mUMpcdwx95OEDeF9mERIhaun9NzxFjmnDyn8qtXIbHdQR4ncbEjuTMi7"
    "BZ6YgAsYsVykZrxbj90iHNuqndK7UGPBrbT0bDhd7hpsn8p09hMSh7bgGIxvUbjEnquH3Zq5jJKR/bJr1bG8CESw5YwBShqgLDvt"
    "cAPNzI34rggmsl0WP3KVptmcdcftOcbVhTEJ9Dtz9eSUiX3r0rb3Rkbn2kploYlDd3d2N7kRS7GVSGpaqpYyJTuWpJQ5f9uOTzBN"
    "U/K4OijIB2xE7NHWcH3z9vN2qiX6oDRp1bDaDcnBY6Swqo+ns/BITozyFzKz2RtTl05mZk3hOBnbChbCwlt5lkxGwDZ7jAl4O53u"
    "1JmQm8t7yXCYv+KJkdAGalkNjeX2qCGgxIQ/LNdlcXxD5eiiPNLrXjOsVbcG7bKnkbxIIDU9cind+G2UEBGLy5d3oQkLZbujnTkZ"
    "RRCxoxwjwty/bacdBSvJKsB0ofEOaSVLcmXMtOzS/jBLH441dCSNmhWnZKy6W/1YPVxK1AmdpbfWULos691IYXjB5M2VhQU458z3"
    "yEgeZU5+KFKvycvkZOTOynOsikmgHzrGd+U1Ng6XLxSuNUSl2SlXWLyQEFx6cVpL64QyJWF0irGK0aM36AXVl1E7bAB8Ai/P+Cjr"
    "d3fDrghVfaS1gxCPYZbtSDZWKaursz9gZtCxUiTECGEddqnaeV+aNKO14nq0yU+jNQbSzE16shuVk1KcSx7HOwhkoZFEV+uBLBnb"
    "mTfYSKNQhq/N+RhwCyw3T7u1eijbj0Y7rKsX8y1WHvbkgaC6t/XZVCyvbW8ym6ju9tfakJIKaDG/sHKNbAVAMMfsM7eKn8qaVq9f"
    "rmzZQS3VxkHEvmasU9Udxai+ASby29m1dIo3ab5Rv1hmieVmQxkKgUwiseWJ0gKrsalNR8gLFIWMADw+boSouG/ibDRXWndK8f4B"
    "wGJlJEFMZXs347XtwVxUs+GxYICu5M96iclx2gYGGq51V5V2/wBlU5llHYknB+VhICrGGhLFyVXtDxO1FJyDrNrQqcupUixhIouo"
    "tLbbOh7fO6IKxMhS3igsGhLS3ZkpGh3OfV1GIBchB1vjMBiUu2ekcMhmL6tRZnWcD8bgImpKnm9lEHSdRwHknIUzyioP6bVWyM/i"
    "GxgrpFIJU1i4m6LITAG01zhWi07liq1IVt5P96xk+sDhDCRDbW+f7CohX9wLbKlYJwgzcoj3e6PoksJC5oGx5zg9V2jcSqfa06Iw"
    "XaLWsgH681UGrEJDI+NNJqNSToBpmbruz+n1fh9SLQ5NePVoHZ7mtVNC2ZhYrh1a7mYRO1TbQmIK5DrlHoPhJUndcANHy23shoyb"
    "YXe152bx6D5y3l70RDTDp7mNq+qdWI/iCmtmkOIvU1MW+6TVkNLtW1wAkwK4ExOxdfd285g1wQtZIe7SNiqfVFbCyNMqHS4lSksg"
    "hruZKU8vFw7X2i3I66EFqLX9IDSrKg2Mt1P8/lBOE5gqZwvNWiJfqLaGrdaYnCbGNtPrZLJFVg71wq7cqBWjDSrRc7KEmO47GXgT"
    "bcD8NX7Nza1uJQOknVJG2YpxXIf4U37rOlPq1FbUJFqqrnpJzlXY1g3nSjvRVrZEgkiE2nCUBMrHpDCGvdtyuxgkiKh7NdVTi+Ej"
    "HJIvnUp0EFDAbqMshK9yvJZHqq1CfXlmm1EATLVDOzjlZ5WgLGqNSgV23J+NnYZAzP1u56IQ4K3RBxlp2/TC9dD2XIiwUJHmz1fC"
    "uKJhtxI9zAsmnyoQVF7T88PpKKp3ucg+Rm/N1aq9u2Tiev6qU3DZ88flKVIODZPauTN0ItIlr2niddTTExBvUXZ2wy1ZQZmQnUVR"
    "dnvm+HacOXmrbbMQdYtX4EXnkLD1I6IP55VDobOIt8xYxxObC/h6UTygNpQrrZlfb0Z7qUyoNRxfJbIKRbOhk7+sYiyVTMVuHsJl"
    "C36+MRXBG+MIUHILL1uF4TE391qWtVwoSgm+OS6eUmurGTcooRRFVO2aak1mkOZMJL/pIlUOVKPxudmuZSaTeRjDWLdfb24OsXZu"
    "cThtIjqLd2J1j1lsmgRoYwjWkofx40ZQaEfchc/5jAWtnG5O0cUouzrk64dsJ9pRCcJwzZNd0FCo7/qjPSYlNoswye2y/NbMj4IQ"
    "0gXYVIhILPF18uhbgrc5d0alZZzbuvpZlONYszzwE1irONULyLBIYuKBHmPz2sTR0u3pDiXs7m4TRphsdJOCVSwRnRaHLn3oeW4P"
    "GHtsdOlE6/qSmx3Xl/YIKm/RhkV05BJVFeKhUozB4lM4ej2jdb4aSSpAb1ydgIQfjwSmO/Yy4yg8aVAw4orTbVPktvEVVl5xeIHD"
    "Tz3nMJTofbwOjzx0X8o1k8e9kgX167jUb0sjqd0YLlJ7S+7HJobrd3vy+czHD9wmppdaFyMVORTXxik/zMvWJhJGBvlatlUjR3D2"
    "pMdyJLfPBD1NZJPSvWqBSYuoPYtasjgPyeRc1RQbVNsVWB/2DAJCdoCCVR3bFiB5rHJEO2ndnJF9zXZTIzHlrNrXSDdzsIr+AhRP"
    "Tv4I97zKAOgdUrMrrO9Sk74vgW1Z549kstLckJYwaPXzB3IAERmDZGqlrEQF3WyFvdQatosAfG2l20d5nyd2R5PrTZFJXzG0LX8m"
    "jYwY8sOXbDGomlLDS9u/gJV1R5mvUa4bnVVrhS08S6UdJFJeIZvwrJQkEjUYrs1H+WyiLyiNCOfmEoSfbUcjmaXYBHJ8fb+1N7zf"
    "qQxOYMcoWw18lETSeo9ZZ5a6nOwASGNl3k6HZC9XVsAcJU+EMMnqUX6+Ck2PnVjfypJrrm8BkQsLVLbpOZuls+2TkpAWqEhX0okk"
    "sMgBG3PM0GPYRDJUtUWCKXl9WqcySq3CJSMbtdFBl9qIS9MhUZsUxnTbcd0+semr8V2lgVglNAfqIN5UayF0uxV3LjyHQ93MkJBT"
    "rIMJjWqcSxe0ei5DzZHodbdpy9B2B3fNM11ZMlh4U9fFjb5ARM6VMkhKDjrANWUfypsUPQuq/6A6LuyPgAxGrVRuSoZDXnbamXGq"
    "F/gq5fU3oROWi4qdid5J0XiVVDBkD5x78G4KmXWJ7EYP/GRKbDAyhAauAADELiKWeTfGKzSi6UMU2kegiT6D+Rybyxdm22q0UuIH"
    "8IGeJ289bU7Z85TKJWvzFBOd2VB7iJwPak2jsFmyjMlNI3pEywx5Ig6AsDRWPt9sFsR0fDAgx5nINXBmF883IsVBkjy7dHG5YnJ6"
    "33SzE7Q/jZe4PgHF8Ma108iTCSmeypPVaIrKOLfKoojgUn2zTiWS+oHhBobtJFLhRZKAaCA6QOfNvKBSiethU5bIW3OZDTNyabXf"
    "TPq8fRD6efF4jtd9ajNsR+SpBuf2q7hayk9w9Hxum7mzmz7quzx9uaETKIvTPnNmc5X0cY9HGZHyrcbEWB6qhlCpIpCJqx4W9Y9h"
    "Q5p7iaPADLA8dorS1xBJ9UuTG3QU44Vlz5/IZ23IIFRM8ranFDw2iyOkswWqaa6d25+3m3wMZs1+zy4czNVN7Ea19t5u2q3xpals"
    "BMsaaBADTrj6nJEB7bhCjqRACRku0TMgY9N2LxgRTjUqTMIuREWOJOBt/3DQpvpgAF2qk9GoyMbP+AzdTYj40cmHY9f81D5HW2Nq"
    "KWPbhX0ErudJqpc35ug+SiGosFY2g3BKrgDb7CBXyxilWYyXg+Z6syDQziacmpKOQRcih2M1sPGiZSc2EU5MdcNRZ3m6qh3XW0Is"
    "Hb0y+fx0x3bBWXQV1Xd6hG3iSJ8s03qo1BOI0HTaMnDYa6xzRDqyuLU87XLVOdxu4+qG2Ry2LsUSuVys65iT8uh8klcFhGPS+3Uh"
    "XRtE1gN8m1mNasIN5inOpEbHSraUiHZPfiZlLXUhk4olOkM15FHynDQOAspnm2hWnnaT8vo6ightuL8+1y9xcI21C+cQv2KF2Hya"
    "Tg/O2yWdn9lhLVGnR+B4UrTwKtOTgKtfz6xcsJ+gz1E9OzUW7jDCVtakOJPYdjJ2uQxqsbyrtEqlkirayVLeXy2TRXEI4Mnrutq1"
    "1h2/23VrUTDUKHXINVNaMJNa5AQnr5FGu7pgqWJE40oax4+WN6vmRuIoEigSCqJBaaKOUq1Tot+MpS46ALb9LtyNbBJZXoDCx06e"
    "wAEkfN6s6q1QLpGMYEwQXcNkrTqwW+noObpdhnU9nxFKahKwoGLOO6RCTneFArq0CDdDxGQtdlaKXq+Bx0mvctzl6qfdzKPGk0a9"
    "3c0PxWFTObNQjJZwAk5eBk4hLKIUA6o1n5NuhMLkJ7OdHtSJWi1BV5d5WF0fqBDjlI9ps4jRdrFlrfu4P82PBom4gFxl6UBI1VLr"
    "uhP6XfFQkPX0WTBF1ANr/QK26XNLf4RVUdhtdk6wJVwdsEVc1EaiozcOk3zzauwg9SRQFb8eT5QmYN9B3HEo1IV1ATSgcA2akaNl"
    "Ub3a+8aWqoybtThy2CebpxHZvbrCgVQSA6AFedF610VLAzuhVLzsUVsfF9dsbgVmi6qS2/j4tLshUcmqFNuFxDoFiBizUcao1kzs"
    "W8UjnmPHLoZruTKjaDa/EI/wIN0r1DvLEzVuMFg6fGmJUChpTsd+zPJjTgGcuMi+ILuJk19Y1HIKhJcNdAxpESrVPiE9miCu2bLb"
    "uhxqkBrq1sboct7flETC7IKDTDxS6XGzyJ4nd6E1S0OhY6TJ99I2I8duIAig7f0N2aJ+0m6w3TEeqyjzAaJeBgM3Yqm5mHcLT5Gr"
    "G23mKkm1lePLV0NuALhYbORyvH9e+mZJ0Ft2/8Cgc5gbLqmrrovLzcprN5l89+r0gOVpNJyKLoKnr6NMl1iyJjIF5DBcLHrQImEt"
    "qlerX3WyyrW5neQincRIKzjxrllzzQuTU2YxuFxgrJhmusXRgMg1DtV9yGl12wUzDG43nl/392JrKIQcBM7X+kzPqrq4bk7Vbqgj"
    "h3wy165v4AzFkpcpD3mrOEStlxG+nOkUhoVpbW3jYzxfdc4Ckps0qiTV2QHIbI5S/nqYZvl05uidJ+aVjMwSHSE3wVZAowmWNYBu"
    "CL4RsmwA2SwLDWAuTIvj/gaI4oOzU1xFx/lixCopTbi2b09SA7/adnPnTkHIxpvNcMmsCK0Ky7RO4jaIL/19JncURwxMO+GuH0L3"
    "9aLtgSIpmF15ItZiFZtZZWNCeQzU2b21z1bC3mlghaAyYkeObmYlZa8rNRG6eSdoHmZ6meYEnfeURBghxsf67Nhab8fTogGCEj9O"
    "pFu7cKaTwkLL4nKwig7tLh9ezNVBsjxelafGdZw8R2bDjmXUIku1WRkEtS89gpTCarHYQNMNdyBn2eiB3ORvhKo1h9DkkIBXTkFL"
    "Ta9Y6ciwGGurHO+tz9XVPJ+LnkO1oTQol7RQuDBQxu5BIlL71B5P5rTi6HKZ5H2WDoqJTpqyV1ozg0K1Q1Yb+9oCP0QH68QQq+6W"
    "JjFcXJrH9mpGJ/PYPHVunMRLj0V32+SqOsT7xzhbKVBWjx42Efysd6rCbjFeUERLzYksr6CiB5RYT8UorRCW6aFtm41iq7SPzNJC"
    "Jqge9mHADA/8vZUp84nOZB3dG3gBSRdjs8ikmqN4pikivc1SojsHdFC/7otn+XjI7YGtmMAyI2KWb7o5kRoVGO7mwImZdWOAWNxx"
    "CeRib9FlqZXbyPKBKx1zzY4zWS7T7Op2bHcKE3KtLpszf1UDkAKNTDN2d7+WS+lDO72qbiPwNFt1uvoURSNGYSsf5nsIVE9AqM7V"
    "WK2obE/8dnCahM4LNilO8pWOdzQhsjA7Kt0FNyZ2m6Byb0ZYXV2YjVuyuYOiTRoZYld+tFMk2VsKSH+4h7QieGyHVQYVu2opnsWb"
    "0XBhcU24QnPTNaFpys5FbuwaaICIEuuwqWjFw8MHzEY3kxGmpNtw9trZ9wyNl/LJ1DRZm8WxdHTpN9Az6Z0Oa5ljeQHTXLbvj9Ih"
    "kiEXHYmvsPx0nTrNMKmpiZOJq+2XjfYtMr6CnfQ83h6Nwlw2V02SQD+m4Lk82zuIwt4vXi/1UDcX1cKwuZapwnik8+62hScrdSgz"
    "AKItSour48SBXWzg8Yq1j2bIGqDeoTaNkPHojWvupFgihpOaVVwsjG2Yv1wnED+mzSNiRItspOXgzeGWwAzfrWKccYOETHfT9eMe"
    "yB27cefUXdUup4ah90dBSzNaE+U2VNueO9lFNJKNp2TV2Rmc2V2kAWDbGZd1a9SLX9hdJW025oMJwSfAkZ6bTY6zVCULEAy0GxfZ"
    "8Y2Z7M5dygdPVZ/voXar20wnJtoFuYT1Jemee4IzLSNRuUFbQ5YqmIvzEMF4vE604U7RmotYcSylRB+aUTuFWZ/iDNFQWcxN9TJy"
    "z0djLp7e185T17wWdvQAns5GyKkbUqIxc19uRJiTF8Y3YSULObv8ZGxnUvMSMh2gWq0Hh1EouU9DkkPUK8cTZl0vYGLCO447GNDi"
    "SbEYUV/vGj3VIzpQLHTZZpQxrzJi4TKD5kyPGjWMcsKC8mJn2evHgEKnEnS7E5QbF4TIGuum6z5U6x0o1ZbNuAA4+eRablI9slTe"
    "i7XOLrNXi9t8IW6GjTSpnYrOBRIrg6D7H691q7Efr1m2JNxy7onCFe1yKCuXeBFMH7BVEp1RDay0XNeapRDcNFgEXdVmjkqjx8wk"
    "yF1553BaSNWC1vUT2+LhfIwOtBmeSI2rbD+ma9mZYOQRiCbrvu5CPHxSSmF6p+OeJ6QILuXP19GtTymeZ9NmvBwdxsSqEiO6yiUi"
    "pcZL3bllklHHsCONYSySPtNkrBBRZDQxu6TmPA2egxi6g2zGFocWVjr3zTEIRM/YRClrZSM+U4ssVWujeEVlenTiml6mE2gdUYaQ"
    "Wk51JqlY0Rhd1pVktVmKD2t0KzHHVtoGUzZEL4Rh1XkGT1BBPsEQIHEgzGIet6Iiz16hI0IqRxK+0j03uU3awCjJXLrTZsNSdUL3"
    "NcaUjhfnhE+rkXzKbTZKvdroYGYH2fwVRJcxt4dEtkswPS2R0HXKBCYktxYhwBmlNpY5XC2ycVl39P2AZCsMaW/jbaAxFRsdyEoP"
    "jK0Ty50PrjWMsZvcGoVjoBEemKtybgdS055+UZbyspYd48SVrc7MZiOS72ZqvAURYYK/hePpyVBY8SO5Urb8UqjfG11ixnQrnfOz"
    "9Y06sVOwSyMJKYZY0AHElq1WLhFbpId2Ze8Zhs1n5vSkfKQEn/BmjkUfV5LdKstrORxZX1hYajnWQZM3Ix3WlSGZCKJDLpmjy+sa"
    "jvghZFaQ7GosnY5PBwgrZPHQuB2Nx3zZupBaBqzFRKB0a5Nk0jvOWpdtbj4/RC7t8RwekEBn6NTqOroy43w85HpZlcN65rLvrOPm"
    "Zad5adSmDmI06VWANUGYUaDgprr+bd4OA0LIQqsWejwN0IPCNjCnuGN4J1+vhQPRC13pFsufSDHmtadqh+i2J1G/V4ppw7i7pqVy"
    "t7Baz5Ct2vIz3DpZWUDb+UiWcqvFvBfFUeiozQdbDsag4SAyqlj7a0HuRa5nPqLOcXZ3QQOrOqW5DJpNqNPiVtBTI64tTM2Eslns"
    "j6Ps2Mgv47nhDiksI6B/iDZ7Wg1ZlE/xQf3S1y/9OaEkukeztc7sItQ1bltVqVyZkbrmxRa5dl+aDzHOO/t6t3i9hXizJxIXoMud"
    "jLVY8UUzdAEplXXGZTXr9OSxDOevJ+3cile2rsdOYUanVcIaLOdT2ooOmcYUnQhbsJyoCDTm7M6JQXFiqhufY6OxHZPBNa/uafnm"
    "yQmLsfLKHoayPjQgyke1FsN2dnQxXeeZot62BvI+awDtzbqcOgGz5AGNrVbWtdzKJqzJ5sxGsnAin5x1UmXl5Nm3XbSkWL1FPcdj"
    "7DC2JshWWdNavH6CD3OTnCYsrNLlhrQ1GSQaMa7By51LMosOAy7k/GVxcyy8L4UnGm4USYMnZCLUrNQ7kTCdlwgH0pc+imT0ZkFv"
    "0EClut6Dl8IS5HYZK10qd6EhfJ61L1Jk4S9zIRWJlZxiR6tMjzk4X2rnrH3ycIrv1K5zsxrz8Kk4uBYMUcGzQmSpFKslqHim4BPK"
    "aFVFHcztUISn4kO6mrdkbppIieIKmY2SprVmx1RWPhSLpdZpXD4X2qPQkJQucGzp9JfUZrDe9zLL1X53W3VWC8b0GQ2Z9KYb1WvO"
    "eHEnxuea0W7ni/lWflo+AJzvSL329Ritp0rMTuyOiAIIAUpcN6BkeSk2yf3VsZzzuRxfcRrWNxDSuWqRxlazVVgv0su0C0eXUSo2"
    "OTSZMeMDy9R1G5/MDdzc5M/HrI0vr9hCzCdNkNDVBjXIKKazApBiJeQYObDtBk2s0puUvU5kiZ5qEN4qXRcHceccIU9sNP1BOd3n"
    "nYy6ENsNM1+gudPl1jLkBOrZhcp8cRyPGDWu5AuEUtTy01D8MC90E1WlHp7f4BnS0K/4EtxO6tQ+cEk/Jl7KWdsp48TRnsu5rHyV"
    "BSJ9C+dyqnBphtFk2IDPfW69wGfu/BqU8bnMCvKPSAGXm3hfaCRuo4zZiQ9AJ8WRDsyXmIEq5yMdGErpa9VPmpfZaR0dIWcuWtr1"
    "xVlVpqONXqufL8lF5zRrT6xuu+XVV93QccQhmbArtUxNBjqbcsvM2EjLrRRafa0kpBnNhbUJE1fDsRIFFfdERRCZVEOgWuGM3awL"
    "GFMa6G2Xh0KVfoTiblMFUSQWzvNJy28Z2/Y429xNZG5hOodW4zTItoH6zmihvGlo+zKJ2j0VXx03IybcPmPUKNo/Cycm0mKSi3It"
    "S6WEDV8crzPAJbZIjbRsNlm/IKfSKFPQEdgeUjvUEGMTdNQ/42yFLp8Vy47M6+wmGekr2zPWwNfNNX0VJ2c3nBCApOjRPt4tVytI"
    "pbZZZPC+S50ttDshTmazHh30T3XWqIOZfKZAJyAIcCQ/0gTWGyo76o+CM693WcJ1OMs+DuvLcRpgLP9CFSCAUFftGrJuFJv+Fqqn"
    "p9cQeMpHcHNJdmkeH9oricGSi4YPX+F+VAfnqn9bQmFGXqvYJHOUolBhXshyzf0NW9LWND4JE3J4Us1AMYRP6ZFN2SFqDHqym9lY"
    "c7LfxDrSHGXnzra/jy+ArjfEEqUqxQ7ZWGhwmTGeDmZEzh6tmnHbPO4LjoACLDoL1WzlZstrpBgWVvVNuEhF0ckICRsJaXFlmlbu"
    "ghyAJFuB6T1J0VmmXwPFoTo8JzNmqwDAvNlV1rd0Yz7DOiOr23X24i1XRwnbncnZq1VrWcwolrocYsfxxUgRWPHM15Rpphedjtir"
    "Wo6t/TB10wtUtaFXO/a5LbHZqr/utUKdaaFZU7fwfLtuAMOrMtrXJRHepAB8kSQPtdkmt09hNMTc0lmT3QK76A7fSOnzSo5viX5X"
    "DtK+mEFKojim1npmf2jZIUrN1FpEYuJhhcqoMKivrAZj2Eh2IHTy7BnyWtcqSqVT24atugZFzA9lBE/hfr90vXlp+XyQi3Gzyswb"
    "jr0wtUqpQfUP1d1UidD7PAsKWil/yWWKM+MYWmxbp7xH3ECYmyEqDsDrKUPvl+GKYR8xkBukoAtdpe2UYl8BfVVYYnoBWXtR0E43"
    "TKwRsiKTip/CWm4NwzPQ+AQCC5YVoHSrVuYnXGg8NkYAT6HrZMNZZPrdQoQmgQjQYsmF4S4hHpDI6ICaAsLaxs7VpMtScwrlY3Gp"
    "lI4eQ9F6OV2k4r0p2FErjHuedcIms60YQKN+BomxGz161ABgyiYsx7rwQG26pTRkkMXetmi4NROeRW4j8ITSaxg24/stGhnOa/F2"
    "icLrztbO+mX4sNpMWRIhlvs65xfEUBt0unYsP3Gx623c6vUWXcjD8N2iUTr4h54zYMYtV8wxjOW0CtWQ5GmQ3o1vThlEDdwH9gxd"
    "nNJtrGSuS0UdHXurETa8HaqlEOYWEwtCIyeJIQvzhImG9zqOTA4wYIcu9q7kaVlH8vpLl+aOO5zIKipdwqHJbka0nKGzxIxpWM5m"
    "R1N7WzkFRYrhVL3idL1YJK0ainU2SEEAcmXYiZYa57zvlxYncpFRyovOGc4bQb7NZhM6Vad2lL/gnZw2qJ1KOJZRrwBd7PRVfxEF"
    "orFtlm8XyXz+3IRPWrkXk7VlhtRKBaNQPuqRVGVzKotj/axcoppEWyOl7s31IT3J8irIWNFyf35kY81YZCEJ9H4ADChwqNaQw6Rg"
    "li60epNzvc2RxTl9UYAn4M5ClL23rba6RLxEeLDYIpyWFzos6PpobDPwLqyYBfWcdSifOkbajSMTjeKizZCzpkNeg0pjBQsQepnE"
    "dyyIpmz3Yhx96WJfWXt+CXVMR6yiSMqSq8aYz4imwM/qFDPFiFJ33OM9yvIsblgt2kl5JBMdLmvwdPx26S3QQtAxFbm1s8v2TXBm"
    "MSC1nkezFtLp77VuMV9TdlcIKu7KEzw78tJ6udbljhqv9JMKpWirtQoNtRi9OBdmUvY0agkIAxBDj9Mbrdn+WpqxqzWjAeNjtKw3"
    "vE1zh9yGl1Cap0o6YB7zU7XKVe3OLsRcAgcYFPtLsTGbxlrpy1HN8Gw9s/L7EH3e26mjmyh1QUGN+bwTA2L5mTO3sVje58R94P4e"
    "0qKH4HQdaakE3GduIt/fpo/n9WZ/bkhYy54pM6XYd5BKQqplK5wNxUu+02yljRIgtBi6tffivcFNsAyDMfVjuLKvhnh12c+dJns3"
    "L4M1faUY1f/4AwT/fP0+9zfXWSeL30Dwjz9WYxTH4SZ+/0o48G1MUzDTRClq/u37y7cBipPE4wHD+/dPhiTx+yfa6dw/aHIA0+T9"
    "CYEpiqQfIF0Ufjy0UYrGcGyBUt/u27QpkqBX1IRYdUkKW5BEsF/q8zRGo4Pxg4n3bX/i5jMT4B+rJjxGV0MKe3yb/fyK9ddLOgd9"
    "Rf3rJZWG3vCDQR56IxIAQ9D1j1UHn9ArJnDKLv2FVObP3M+k0n9+IvUcvJLK/JkOSOHwmF6NaXQYEEoG50MCSugKIYk21kKJB6vQ"
    "H3/8IUrrl5Xp8qoirGxF1jnHtSRgzVka+Nfjy/bfvn1DDM3kBOflCfayVnRZskxL0Z2XtWG9iJIjCU4w+cK9aIplGZYkvvCuooqP"
    "nwbciQiGqzv2/UiKI2l/vUAPxPvzi6K/PL/V/9gMIZmHvLsoOrw/dEhyjH7R5oeC35T+gf0PRjMMRDGhHmQQknzQZVC09YoJXh8f"
    "d4Ys43Dn5376P2XJAb45iirZAfzfP8CXYP3vH3+9b3aHvy/fEe54Py3f/5T1i244L4qt6LbD6YIE3BG+v4iK4IBfQZ9C0gMputKX"
    "hfs2O+n4ENM3Tlc0Tr3zH7iYef/cKbr47Te0PE51pUDe9w0fRwlogL9ABQw+AQPqTxX9SulDfX8/YH+8JO7+8jsw3pK43WPFNGzF"
    "UQz9rvK/P8R5f5KsuzwDc4V+gN9f4sDH6obTxS/CBn88iFlSYJOfrUSV9H9F+7AIxzVVCbANy5HETyiurhrCThJXe5cTLS442xf8"
    "fyLwHLydDXzo5m10F+H7qX+l8CrBu8X/+GL7r6Sf62944JtjvnvkSlReTUiV1s73F0uRN2829LYUyJrj7QfA39CPl+QTKHgEv4IF"
    "+su8xF9h9Sd8Kjh88iHYJ1Iw/hXLdjXgjsUFoPzz+Nz3F/5+jJNiPgllfrwyFzy9kgis7LGU/vESqr4upj+5yuct0p81/rbwJg3X"
    "FDlHWj1+irQyLWMdGDdg8Pb3F9uRzFdpyKrBc+rLL/HujZU76NMtA5PKfn9JZ8GXwIIeU8BjsVZ9yRYfc49hNAB5qQbR8pObPRl8"
    "xo3Aqu5mHvDxYeRf7OlDCE+ztcGXykv6t8RMlTtKVkAtiK3AO8XnbEASelCEwJ/V/js7eSf/++hu//2k+tlb/gkyFej7K/S7Yt+Z"
    "qARB4eNIv8s2mqIDxe+/WUq8pJ70JPUnitn/giLnA9DvKCY/KNrS/5hG5t0N11YQmleWqwNBGgwc/PvLz1YXJLqxpKovgWW+/zxO"
    "VYKBommSqARmqx5feClwGSnIkg8L/g/7RfLNIHMG2VJWXec9WQZC+JWdwGLuuv9N9RLICfqtMRmWKFl301QDkb5y/jQojbN2kvMl"
    "5n020icieHeE1O9Jc2oQ6sXjPaF/ZM8H2t2tnvgfiAHZTznwsfr9wdTT8963fOyYecw9xvcoFjjet3th+O1rZnpl4O8nXOrHj3vw"
    "e849jvg2//CZxJuWH171WAli0cOTwDe30/VADe/HkXTx1WLvzD1/qQg+FR5Q+40S3m34Loi1+3DFB/S9OuCCegl4xU19vxMHfxP9"
    "ql/wko/NvlQb7/J98vP3J/Af/6TX3xYkwK+FwH9fPb/g/l5dn6ZTPx5M/1Rgf6Hz362GHlmz+k73y1qQywMM5/jh1P+k7s92eSf4"
    "mg1ereBXVl4XXrN3UNG86ez7+6ZfWfkadn5L5i6VIN+94T+A7c3DBj8FfkvxgtDx7aHM8xV8zt7B3qae5mspgnR3dOAD9dUUvmI+"
    "Ad8nn6X5xnjFfs1Ub9XJ9xfg13OCnwT15x3KBj7pjvM4RX38mPiLDu4MPxh4kn3LYoGNf3bZt8UPHX3WaOCK79Q/sfM5xryDf42I"
    "vzWmhyTuLqcanAM85fKJi08t3VNdnxPvK75hPVn76k1PwvEPZuIvn7u6J7UvGIkX4PcBJZDPmwLAO5mfeXqn8sHWqzL/5EwziDIA"
    "8Mbm91eNvsvtPd6/YnxKkj/DvuryFfBjr2eUf9vq76fb/4z84wP+mYD+frPMH29+bP/9Vwr68ZZqHckKdM2pK1XZu0pQ7wU4/5Z1"
    "KelemSQDOooXeI/NraWAZV1y/goCF+dJgXO/2EFmfljlw8EU3ZN0x7COL5wTLHF2ULwHJ/icfh8RuPKSL/4++/1P/fR58numeuZi"
    "W3KCM3Ou+lPo/jnFfhX557j3JS08SfzPsm76H8L4g9r1S1x4BPO3u5oP+ewflvIv7v7FVQPg2gv02PRzCH479BuDzyPd6+XUTx79"
    "XPkn63sY3lsPFbCz4gTh/v8EbOUkvdrOhlPXAcP3mZf//M+vjcffwGM1ec/Wb09B+n/M/jrzAfcVBnw3as3wpJVjHDhLvLeP318c"
    "zgoE9P3RntuvDPnfX+4SDNYfQycYP0T6hP19rA5E6fiBdPy//jEEfGPQ8f26BPCffAa+D37Crf0rLgq/4SZ+wQ3i7MvxX3AJkqK7"
    "D+T7we7y+oJc+1fkMTn5hJz4QH7oq/owjqfwPspPMwDXA3j9maV+iWzBvtA9N+h3iT3o3M3sOXX8PPUg/Ld+/PG37j961m84ifTR"
    "1k816JuxGOaXe4q/7/dN42+/RrSn19+jxkf4InlbsrxHlEuK9xCm36/XODUp3dvk5IPghrM8yXb+U7QM8z/vseyRzixDVSXrPWr9"
    "/3SO967uS/x6tqxvrcBbZ/hWW9xD2+cG96do97TWu8ju+fSnq7PPLfDvVfe/uCr6ek30FtcV6b3neeX0ifNp/dvXrudj4U/Jd+4G"
    "eL4+TGr1Ub7fuf6433m9K/mE+W6fQbS57x/EduDX0BMEpZ/rpI/o/rB18EtsDxL3652kpLuBPILDfJbdG5j//f0q8gMuQAT/IRl8"
    "uoZ8huDAZt5vCr8dFUkVV66uPK7F3qu1IF5/ygZPY/4Uiu6Oe78D/tISBhJbWdzh+6ek+3pZ9C7L75/l/6mWDADuwe/tyu1O5tNx"
    "3slVPz1/mOPjltDg7ln6fmn1qRb1XhuBB3venZt3/D8f95tBQfuxz08x+c3SXy9W7b+DEBFEiLfAcvwlrrxHmntH8EIEXf9njTw4"
    "fEuD9/Pe7wPv9vNTr/sQ9V3S31oUOfz245cL4mfWe9whfj3RLxX6z63P73P5l4L2qdf35ud1/DXBf+mwn3z81G897nb+D0zwH0XT"
    "hanpPdN9lc67p/0ZlNLCa/79ias3NXwl/Uy7r63HQyvf7xfxVZXTeJF72f/1uEDdP29a/bsInuPUfXz8qdN85/O/KgW+cvbO/b+w"
    "9qkr+soe8K/8BUXS43rkDvAzs7/I7Lnh/8mR/sUEPtS/lqxgEPiNtXpnJrCDNhd4EfiPFoCQOI4i9OrT+74ff/xzS/4J8zVhf115"
    "r0aew9fQ/Vr737n+ml4+V/vgH//gmW/O84tf3rHfXezfeubEk9j/vu+/FxKP8P2Rf/5t3/iDq18a5dRj/V4KP8f/QuJryvol+jxW"
    "f3xl7Z77Pr2Xe84HRhSM71c6gc9+Qvjz/ioFsCQvaCmlKm25Evi16/r7N13CM81/mnnf59GVvlfZX6qqjeF+rql+uvv+aNzuSJ9q"
    "FsWS7MBPRO747Tdd0U/1xt21U9BrnfHaBn1/i7LFj6uToO7498aoi1Hot9dK57VCPb9VV3+9mXoQBL6/vFZVH5Opv+6zrxb11yvl"
    "h1yur8UtJ0tPwdxtXF8rcvWe4356E/P+Jvr7P7yVebTbz2j2Rc73+U+i+v7y6TL2447/rV+/v6K5Az6b9+qnff/9DcCzqvnyvvz9"
    "6vW/euX0hYGgjS6kfnmv8/sG4I8vEej+j/L+FCXJvD+8nvDvx73uE/BfX0A8If6b9yafzeC5+sf/A3aVeD0="
]
_AGENT_B64 = "".join(_AGENT_B64_PARTS)
EXPECTED_MAIN_SHA256 = "e92745da9546743b952a220585a875f254cea7b7ed79b10c465f40c941093885"
EXPECTED_MAIN_BYTES = 20487
BEST_LABEL = "c16_common_meta (stable 8-cow / 5-sheep / 12-hand schedule)"
BEST_PUBLIC_SCORE = "not submitted; local paired-seat gate only"

raw = zlib.decompress(base64.b64decode(_AGENT_B64.encode("ascii")))
assert len(raw) == EXPECTED_MAIN_BYTES, (len(raw), EXPECTED_MAIN_BYTES)
digest = hashlib.sha256(raw).hexdigest()
assert digest == EXPECTED_MAIN_SHA256, (digest, EXPECTED_MAIN_SHA256)
assert b"def agent" in raw, "agent() missing from payload"

MAIN_PATH.write_bytes(raw)
compile(raw, str(MAIN_PATH), "exec")

with tarfile.open(ARCHIVE_PATH, "w:gz") as archive:
    archive.add(MAIN_PATH, arcname="main.py")

with tarfile.open(ARCHIVE_PATH, "r:gz") as archive:
    members = archive.getnames()
assert members == ["main.py"], members

from kaggle_environments import make

spec = importlib.util.spec_from_file_location("embedded_c16_agent", MAIN_PATH)
module = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(module)

env = make(
    "kaggriculture",
    configuration={"episodeSteps": 720, "seed": 0},
    debug=False,
)
env.run([module.agent, "starter"])
status = [row["status"] for row in env.steps[-1]]
rewards = [row["reward"] for row in env.steps[-1]]

print(
    {
        "selected": BEST_LABEL,
        "ladder_publicScore_reference": BEST_PUBLIC_SCORE,
        "main_py": str(MAIN_PATH),
        "main_bytes": MAIN_PATH.stat().st_size,
        "main_sha256": digest,
        "submission": str(ARCHIVE_PATH),
        "submission_bytes": ARCHIVE_PATH.stat().st_size,
        "archive_members": members,
        "smoke_vs_starter_status": status,
        "smoke_vs_starter_rewards": rewards,
        "note": "This cell only builds files. It does not submit them.",
    }
)


### Reproducing the artifact

Run all cells and check that the final output reports
`archive_members: ['main.py']`, a matching SHA-256, and smoke statuses of
`DONE`. The generated files are `main.py` and `submission.tar.gz` in the
notebook working directory.

This notebook does not call the Kaggle submission API. I prefer to inspect
the generated archive and the local gate first, then make submission a
separate, explicit decision.
